In [ ]:
import os
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import copy

# Paths
DATASET_PATH = "your path"
RESULT_PATH = "your_path"
os.makedirs(RESULT_PATH, exist_ok=True)

# Load and preprocess dataset
df = pd.read_csv(DATASET_PATH)
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')
df = df.sort_values(['Company', 'Date']).reset_index(drop=True)

features = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
scalers = {}
for feat in features:
    scaler = StandardScaler()
    df[feat] = scaler.fit_transform(df[[feat]])
    scalers[feat] = scaler

df['Target_Close'] = df.groupby('Company')['Close'].shift(-1)
df = df.dropna(subset=['Target_Close']).reset_index(drop=True)

def create_graphs(df):
    graphs = []
    df['YearMonth'] = df['Date'].dt.to_period('M')
    grouped = df.groupby(['Company', 'YearMonth'])

    for (company, ym), group in grouped:
        group = group.sort_values('Date').reset_index(drop=True)
        if len(group) < 2:
            continue
        x = torch.tensor(group[features].values, dtype=torch.float)
        y = torch.tensor([group['Target_Close'].values[-1]], dtype=torch.float).view(-1, 1)

        edge_index = []
        for i in range(len(group) - 1):
            edge_index.append([i, i+1])
            edge_index.append([i+1, i])
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

        data = Data(x=x, edge_index=edge_index, y=y)
        data.company = company
        data.year_month = str(ym)
        graphs.append(data)
    return graphs

graphs = create_graphs(df)

clients = {}
for g in graphs:
    clients.setdefault(g.company, []).append(g)

from torch_geometric.loader import DataLoader

client_loaders = {}
for c, g_list in clients.items():
    n = len(g_list)
    train_n = int(0.8 * n)
    train_loader = DataLoader(g_list[:train_n], batch_size=4, shuffle=True)
    test_loader = DataLoader(g_list[train_n:], batch_size=4, shuffle=False)
    client_loaders[c] = (train_loader, test_loader)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
in_channels = len(features)
hidden_channels = 64
heads = 4  # Number of attention heads

# Graph augmentation functions
def drop_edge(data, drop_prob=0.2):
    edge_index = data.edge_index
    num_edges = edge_index.size(1)
    keep_mask = torch.rand(num_edges) > drop_prob
    edge_index = edge_index[:, keep_mask]
    data_aug = Data(x=data.x, edge_index=edge_index, y=data.y)
    return data_aug

def mask_features(data, mask_prob=0.2):
    x = data.x.clone()
    mask = torch.rand(x.size()) > mask_prob
    x = x * mask.float()
    data_aug = Data(x=x, edge_index=data.edge_index, y=data.y)
    return data_aug

def graph_augment(data):
    data_aug = drop_edge(data, drop_prob=0.2)
    data_aug = mask_features(data_aug, mask_prob=0.2)
    return data_aug

# Contrastive loss
def cosine_similarity(x1, x2):
    x1 = F.normalize(x1, dim=-1)
    x2 = F.normalize(x2, dim=-1)
    return (x1 * x2).sum(dim=-1)

def contrastive_loss(z1, z2, temperature=0.5):
    batch_size = z1.size(0)
    z = torch.cat([z1, z2], dim=0)  # 2B x D
    sim = torch.matmul(z, z.T) / temperature  # similarity matrix

    mask = torch.eye(2*batch_size, dtype=torch.bool, device=z.device)
    sim.masked_fill_(mask, -9e15)

    positives = torch.cat([torch.diag(sim, batch_size), torch.diag(sim, -batch_size)], dim=0)
    negatives = sim[~mask].view(2*batch_size, -1)

    labels = torch.zeros(2*batch_size, dtype=torch.long, device=z.device)
    logits = torch.cat([positives.unsqueeze(1), negatives], dim=1)

    loss = F.cross_entropy(logits, labels)
    return loss

class GATContrastive(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, heads):
        super(GATContrastive, self).__init__()
        self.gat1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.1)
        self.gat2 = GATConv(hidden_channels*heads, hidden_channels, heads=1, concat=False, dropout=0.1)
        self.lin = torch.nn.Linear(hidden_channels, 1)

    def encode(self, x, edge_index, batch):
        x = self.gat1(x, edge_index)
        x = F.elu(x)
        x = self.gat2(x, edge_index)
        x = F.elu(x)
        x = global_mean_pool(x, batch)
        return x

    def forward(self, x, edge_index, batch):
        emb = self.encode(x, edge_index, batch)
        out = self.lin(emb)
        return out, emb

def train_contrastive(model, loader, optimizer, device, alpha=0.5):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)

        data1 = graph_augment(data)
        data1 = data1.to(device)
        data2 = graph_augment(data)
        data2 = data2.to(device)

        optimizer.zero_grad()

        out1, emb1 = model(data1.x, data1.edge_index, data1.batch)
        out2, emb2 = model(data2.x, data2.edge_index, data2.batch)

        out_orig, emb_orig = model(data.x, data.edge_index, data.batch)
        mse_loss = F.mse_loss(out_orig, data.y)

        cont_loss = contrastive_loss(emb1, emb2)

        loss = mse_loss + alpha * cont_loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

def evaluate(model, loader, device):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out, _ = model(data.x, data.edge_index, data.batch)
            preds.append(out.cpu())
            trues.append(data.y.cpu())
    preds = torch.cat(preds).view(-1)
    trues = torch.cat(trues).view(-1)
    mse = mean_squared_error(trues, preds)
    return mse, preds, trues

def get_model():
    model = GATContrastive(in_channels, hidden_channels, heads)
    return model.to(device)

def average_weights(w_list):
    avg_w = copy.deepcopy(w_list[0])
    for key in avg_w.keys():
        for w in w_list[1:]:
            avg_w[key] += w[key]
        avg_w[key] = torch.div(avg_w[key], len(w_list))
    return avg_w

epochs_per_round = 3
rounds = 10

global_model = get_model()
global_weights = global_model.state_dict()

mse_per_round = []

for r in range(rounds):
    print(f"Round {r+1}/{rounds}")
    local_weights = []
    for c, (train_loader, _) in client_loaders.items():
        local_model = get_model()
        local_model.load_state_dict(global_weights)
        optimizer = torch.optim.Adam(local_model.parameters(), lr=0.01)
        for e in range(epochs_per_round):
            train_contrastive(local_model, train_loader, optimizer, device, alpha=0.5)
        local_weights.append(local_model.state_dict())
    global_weights = average_weights(local_weights)
    global_model.load_state_dict(global_weights)

    # Evaluate global model on combined test data
    all_test_data = []
    for _, (_, test_loader) in client_loaders.items():
        all_test_data += test_loader.dataset
    all_test_loader = DataLoader(all_test_data, batch_size=8, shuffle=False)
    mse, preds, trues = evaluate(global_model, all_test_loader, device)
    print(f"Global Model MSE: {mse:.4f}")
    mse_per_round.append(mse)

plt.figure()
plt.plot(range(1, rounds+1), mse_per_round, marker='o')
plt.title("Global Model MSE per FL Round with Contrastive Learning")
plt.xlabel("Round")
plt.ylabel("MSE")
plt.grid(True)
plt.savefig(os.path.join(RESULT_PATH, "mse_per_round_contrastive.png"))
plt.close()

with open(os.path.join(RESULT_PATH, "mse_per_round_contrastive.txt"), "w") as f:
    for i, val in enumerate(mse_per_round, 1):
        f.write(f"Round {i}: MSE={val}\n")

print(f"Training finished. Results saved in {RESULT_PATH}")


Round 1/10
Global Model MSE: 0.9367
Round 2/10
Global Model MSE: 0.4736
Round 3/10
Global Model MSE: 0.5665
Round 4/10
Global Model MSE: 0.5271
Round 5/10
Global Model MSE: 0.5069
Round 6/10
Global Model MSE: 0.4177
Round 7/10
Global Model MSE: 0.4173
Round 8/10
Global Model MSE: 0.1611
Round 9/10
Global Model MSE: 0.1802
Round 10/10
Global Model MSE: 0.2456
Training finished. Results saved in /Volumes/SP_SAGHAR/Documents/University/Articles/Conference/6- ICAEA -sbu - enterprise/result/2- GAT + Contrastive Learning
